# Demo Phúc–Đăng: detection và tracking trên dữ liệu tổng hợp

Notebook mở kết quả detection PHUC-005 đã khóa và hai run tracker tái lập từ PR #12 của Đăng. Có đủ hai đường **truth → tracker** và **graph → Leiden seed 0 → tracker**. Dữ liệu graph/truth/Leiden/AMI gốc giữ nguyên; notebook chỉ đọc output có hash, không chọn lại seed/ngưỡng.

- Node là user giả lập; cạnh do bộ sinh lấy mẫu, không phải Reddit.
- AMI đo khớp phân hoạch; event-F1 đo nhãn thay đổi của nhóm. Hai chỉ số trả lời hai câu hỏi khác nhau.
- Hai run tracking do Phúc tái lập. Artifact và freeze lịch sử gốc của Đăng chưa được nhận; không coi lần tái chạy này là một đánh giá giữ lại mới.

## Chạy nhanh

Làm theo `document/guide.md`: cài package `.[graph,evaluation,notebook]`, dùng ZIP T02 đã nhận, chạy `scripts/run_tracking_handoff.py`, rồi mở notebook này với kernel của dự án và Run All. Hướng dẫn có lệnh chạy tự động bằng kernel mới và chỉ rõ input/output. Notebook nguồn không cache kết quả; bản đã chạy nằm trong artifacts.


In [ ]:
import importlib.util
import sys

print("Active Python:", sys.executable)
if sys.version_info < (3, 11):
    raise RuntimeError("Chọn kernel Python 3.11+ của dự án.")
missing = [name for name in ("reddit_sna", "matplotlib", "numpy") if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Thiếu {missing}. Cài package với extra [notebook] bằng Python của kernel, rồi restart.")

import json
import os
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from IPython.display import display, HTML, JSON
from IPython import get_ipython
import matplotlib.pyplot as plt
get_ipython().run_line_magic("matplotlib", "inline")
from reddit_sna.reporting.synthetic_demo import (
    load_synthetic_demo, build_synthetic_figures, export_synthetic_demo, table_html,
)
from reddit_sna.reporting.synthetic_cases import (
    analyze_synthetic_cases, build_case_figures, case_summary, export_case_analysis,
)

## 1. Chọn input/output và phạm vi

ROOT được tìm từ cwd hoặc `REDDIT_SNA_PROJECT`; không phụ thuộc đường dẫn cá nhân. `REDDIT_SYNTHETIC_BUNDLE` có thể trỏ tới bản giải nén cùng gói T02 ở chỗ khác. `REDDIT_SYNTHETIC_DEMO_OUTPUT` chọn output mới bên trong `artifacts/` của repo; để trống sẽ tạo tên timestamp mới. Input/output không chồng nhau, không ghi đè run cũ.

Dataset minh họa mặc định **dev-grow-easy-g11**, chọn trước khi mở điểm. Có thể đổi `REDDIT_SYNTHETIC_DEMO_DATASET` để xem dataset khác; không thay seed canonical theo score. Notebook trình bày cả hai tập của detection đã khóa; Đăng chỉ dùng evaluation sau khi tự khóa tracker/evaluator của mình.

In [ ]:
search = Path(os.environ.get("REDDIT_SNA_PROJECT", Path.cwd())).resolve()
ROOT = next((p for p in (search, *search.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/reddit_sna").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Đặt REDDIT_SNA_PROJECT tới root repo đã cài package.")

reference_path = ROOT / "document/specs/phuc/evidence/PHUC-006-T02-delivery-reference.json"
reference = json.loads(reference_path.read_text(encoding="utf-8"))
BUNDLE = Path(os.environ.get("REDDIT_SYNTHETIC_BUNDLE", ROOT / "artifacts/handoff/phuc-dang-week1-20261019-001")).resolve()
DATASET = os.environ.get("REDDIT_SYNTHETIC_DEMO_DATASET", "dev-grow-easy-g11")
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = Path(os.environ.get("REDDIT_SYNTHETIC_DEMO_OUTPUT", ROOT / "artifacts/detection-tracking" / f"phuc-dang-demo-{STAMP}")).resolve()
NOTEBOOK_SOURCE = ROOT / "notebooks/detection_tracking/01_synthetic_demo.ipynb"
if not OUTPUT.is_relative_to((ROOT / "artifacts").resolve()):
    raise ValueError("Demo output phải nằm trong artifacts/ của repo.")
if OUTPUT.exists() or OUTPUT.is_relative_to(BUNDLE) or BUNDLE.is_relative_to(OUTPUT):
    raise ValueError("Chọn output mới, không chồng input.")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print("Input:", BUNDLE)
print("Output mới:", OUTPUT)
print("Dataset minh họa:", DATASET)
print("Delivery manifest SHA-256:", reference["delivery_manifest_sha256"])
ANALYSIS_OUTPUT = OUTPUT.with_name(OUTPUT.name + "-cases")
if ANALYSIS_OUTPUT.exists() or ANALYSIS_OUTPUT.is_relative_to(BUNDLE) or BUNDLE.is_relative_to(ANALYSIS_OUTPUT):
    raise ValueError("Chọn output case analysis mới, không chồng input.")
print("Output phân tích riêng:", ANALYSIS_OUTPUT)
TRACKING_OUTPUT = OUTPUT.with_name(OUTPUT.name + "-tracking")
if TRACKING_OUTPUT.exists():
    raise ValueError("Chọn output tracking mới, không ghi đè.")


## 2. Kiểm tra source rồi mở dữ liệu

Helper kiểm tra bản khóa manifest của ZIP T02 và hashes/sizes toàn bộ file khai báo. Consumer T01 xác minh source runs complete, identity/reference, node/edge/truth keys và membership đủ ba seed đúng endpoint. Các metric/summary được đọc từ CSV đã hash; không tính lại hoặc thay dữ liệu để đẹp hơn.

Mỗi split có dataset IDs và generator seeds riêng. Detector seed là lần lặp trong dataset, không phải dataset độc lập. Metadata producer bên dưới là môi trường lúc sinh/chấm, khác môi trường đang vẽ notebook.

In [ ]:
start = perf_counter()
demo = load_synthetic_demo(BUNDLE, expected_delivery_sha256=reference["delivery_manifest_sha256"], dataset_id=DATASET)
print(f"Replay verification/read wall seconds (not producer runtime): {perf_counter() - start:.3f}")
source_rows = [dict(split=split, **inventory["counts"],
                    source_manifest_sha256=inventory["source_manifest_sha256"])
               for split, inventory in demo["inventories"].items()]
display(HTML(table_html(source_rows)))
print("Graph recipe đã lưu:")
display(JSON(demo["recipes"]["development"]))
print("Producer environments:")
display(JSON(demo["environments"]))

## 3. Bảng AMI/coverage và support đầy đủ

Nguồn: `runs/<split>/evaluation/detection_summary.csv`. Mỗi seed giữ riêng; mean lấy trên dataset-month có AMI hợp lệ, không trộn seeds/splits. Bảng giữ số valid/null và reason, không điền thiếu bằng 0 hoặc 1. **Coverage 1 không đồng nghĩa tìm nhóm đúng hoàn toàn.**

Policy của đồ án cho AMI null khi phân hoạch hằng, quá ít user hoặc rỗng; dissolve tháng cuối chỉ còn nhóm nền nên truth constant. Đây là trạng thái thiếu có lý do, không tự coi là lỗi pipeline. Mục 7 đối chiếu một case khó và một case null từ development.

In [ ]:
all_summary = [r for r in demo["summary_view"] if r["level"] == "all_scenarios"]
display(HTML(table_html(all_summary, ["split", "detector_seed", "seed_role", "n_rows", "n_ami_valid", "n_ami_missing", "ami_mean", "coverage_mean", "missing_reasons"])))
print("Metric rows thật:", len(demo["metrics"]), "| Summary rows thật:", len(demo["summaries"]))

## 4. Nhìn graph và nhóm tìm được

Nguồn: các paths `nodes_file`, `graph_file`, `truth_membership_file`, `detector_membership_file` trong inventory của dataset đã chọn. Vẽ **toàn bộ active nodes và cạnh** hai tháng; không lấy mẫu để tính hay thay chỉ số. Cùng user có cùng vị trí vòng tròn qua các panels, không dùng layout như bằng chứng chất lượng.

Màu/ID nhóm chỉ có nghĩa trong từng panel; không ghép nhóm truth với detector bằng tên hoặc màu. Nếu có isolate không được detector gán nhóm, dùng dấu × xám. AMI/coverage ở bảng dưới là từ full graph report.

In [ ]:
figures = build_synthetic_figures(demo)
for figure in figures.values():
    plt.close(figure)  # giữ objects để display/export, tránh Jupyter tự hiển thị trùng
display(figures["01_network"])
selected_metrics = [s["metric"] for s in demo["network"]["snapshots"]]
display(HTML(table_html(selected_metrics, ["dataset_id", "month", "detector_seed", "active_nodes", "endpoint_nodes", "assigned_nodes", "isolated_nodes", "coverage", "ami", "ami_status", "ami_reason"])))

## 5. AMI theo case và diagnostics giữa các seed

Hai hình lấy **recorded summary**, không chọn seed có điểm cao nhất. Ở hình case, số trên cột AMI là `valid / total`; mean dissolve chỉ dùng những dòng hợp lệ và vẫn báo phần thiếu. Giá trị thiếu là khoảng trống, không giả thành điểm 0. Seed 0 luôn canonical, 1/2 diagnostic; điểm khác nhau không phải ba dataset mới.

In [ ]:
display(figures["02_quality"])
display(figures["03_seeds"])

## 6. Thời gian và phạm vi đo

Nguồn: `runs/<split>/manifest.json` của lượt chạy PHUC-005 cũ. Đây là thời gian sinh/detection/evaluation **đã ghi nhận**, không phải thời gian vẽ notebook hiện tại. Stage wrapper có checkpoint/integrity overhead; detection bao gồm cả ba seeds. Tổng run có scope riêng, không cộng hoặc chia seed để tạo thời gian giả. Timing chi tiết mỗi dataset-month có sẵn trong report cũ; không nhân 36 dòng tháng thành 108 dòng thời gian seed. Peak memory chưa đo.

In [ ]:
display(HTML(table_html(demo["timings"])))
display(figures["04_timing"])
print("Peak memory: chưa đo; không có số giả để vẽ.")

## 7. Case khó và AMI null — PHUC-006-T04

Đây là **phân tích sau khi đã xem kết quả (post-hoc)**, chỉ chọn trong **development, canonical seed 0**: dòng AMI hợp lệ thấp nhất, hòa thì dataset ID/tháng tăng dần; case null đầu tiên theo dataset/tháng. Không chọn lại seed, không đổi gamma/dữ liệu/policy và không chạy lại detector/scorer. Selection có trong output; ví dụ khó không đại diện trung bình toàn suite hoặc một đánh giá mới độc lập.

Mỗi hình vẽ **toàn bộ graph** của snapshot, truth và prediction ở cùng vị trí user, cạnh giữ nguyên. Bảng bên phải đếm số user theo cặp nhóm truth–prediction, **không phải confusion matrix sáu sự kiện**. ID/màu chỉ dùng cục bộ. Đối chiếu bằng user; không coi đổi tên nhóm là lỗi.


In [ ]:
cases = analyze_synthetic_cases(demo, split="development")
display(HTML(table_html(case_summary(cases))))
case_figures = build_case_figures(cases)
for figure in case_figures.values():
    plt.close(figure)
    display(figure)
for case in cases:
    print(case["kind"], case["dataset_id"], case["month"], "| selection:", case["selection"])
    print("Input paths/hashes:")
    display(JSON(case["inputs"]))

### Đọc kết quả và giới hạn

Trong case merge, đọc hàng truth `C` của bảng đếm: người cùng một nhóm chuẩn rơi vào hai cột detector khác nhau, trong khi nhóm nền `K` nằm riêng. Đây là **chia nhỏ một nhóm ở một snapshot**, chưa phải tracker dự đoán sự kiện split. Coverage chỉ nói đủ người có nhóm, không đảm bảo phân hoạch trùng truth.

Bảng cạnh bên dưới đếm cạnh trong/giữa các nhóm detector; density = cạnh thực / số cặp có thể nối (nhóm n người có n(n−1)/2 cặp; hai nhóm n,m có n×m cặp). Độ dày cạnh quan sát được giúp mô tả graph đang có, **chưa chứng minh nguyên nhân tối ưu của Leiden** hoặc cho phép chọn lại gamma/seed theo điểm. Pair disagreement đếm cặp user được gán nhóm cùng/khác truth; giữ số user isolate ngoài phạm vi cặp, không đổi thành event-F1 hay AMI mới.

Trong case dissolve, truth chỉ còn một nhóm `K`, nhưng detector có thể chia thành nhiều nhóm. Policy đã khóa trả `AMI=null`, `degenerate_partition/truth_constant`; đây **không phải điểm 1 và cũng không che việc hai phân hoạch khác nhau**. Vẫn hiển thị graph, contingency và pair counts, không điền AMI thiếu bằng 0/1. Nulls của suite vẫn giữ nguyên support trong summary.

Các kết luận này chỉ về graph tổng hợp và phân hoạch đã lưu; chưa đánh giá tracker/forecast/Reddit. Evaluation đã khóa chỉ được replay bảng chung ở trên; case analysis này không mở thêm một vòng tuning/evaluation.


In [ ]:
for case in cases:
    d = case["description"]
    print(f"{case['dataset_id']} {case['month']}: {d['active_nodes']} active users, "
          f"truth {d['truth_groups']} groups → detector {d['predicted_groups']} groups; "
          f"{d['unassigned_nodes']} unassigned users.")
    print("Cặp cùng truth nhưng khác detector:", d["truth_same_pred_different_pairs"],
          "| khác truth nhưng cùng detector:", d["truth_different_pred_same_pairs"],
          "| tổng cặp được xét:", d["scored_pairs"])
    display(HTML(table_html(d["edge_blocks"])))

## 8. Lưu đầu ra và nguồn từng hình

Output mới gồm 4 hình PNG + SVG, full metrics/summary CSV, summary view và timing CSV, `figure-sources.json` chỉ tới input paths/hashes và metadata complete ghi cuối. Helper kiểm tra lại source bytes trước/sau export. Notebook nguồn trong Git không chứa kết quả cũ; bản đã chạy và logs nằm ở artifacts evidence.

Không sửa ZIP T02 đã giao. Nếu output tồn tại, chạy lại cell cấu hình/Run All để lấy tên mới. Khi gặp lỗi, giữ output dở dang và đọc `failure.json` thay vì xem nó là complete.

Phân tích T04 lưu vào thư mục **cùng cấp** có đuôi `-cases`: hai hình PNG/SVG, case summary/contingency/edge-block CSV, analysis JSON và input-hash ledger/metadata riêng. Inventory output T03 chỉ bao phủ bốn hình/bảng replay; không ghép file mới vào inventory cũ.

In [ ]:
metadata = export_synthetic_demo(demo, figures, OUTPUT, notebook=NOTEBOOK_SOURCE)
print("Đã lưu:", OUTPUT)
display(JSON({k: metadata[k] for k in ("status", "scope", "selected_dataset", "metric_rows", "summary_rows", "figure_count", "canonical_seed", "partner_acceptance")}))
print("Nguồn figure và checksums:", OUTPUT / "figure-sources.json")
analysis_metadata = export_case_analysis(demo, cases, case_figures, ANALYSIS_OUTPUT, notebook=NOTEBOOK_SOURCE)
print("Đã lưu phân tích:", ANALYSIS_OUTPUT)
display(JSON({k: analysis_metadata[k] for k in ("status", "scope", "case_count", "figure_count", "canonical_seed", "partner_acceptance")}))

## 9. Hai đường tracking và hai cách tổng hợp F1

Cấu hình giữ nguyên PR #12: Jaccard ≥ 0,30, thay đổi kích thước vượt ±10%, ưu tiên split trước merge, detector seed 0. Reader kiểm tra manifest, input pins, cấu hình, schema, references, mapping và tính lại metrics từ predictions đã lưu.

**Mean dataset macro-F1** lấy trung bình điểm của 18 dataset, mỗi dataset vẫn chấm sáu lớp nên lớp không xuất hiện nhận 0. **Pooled macro-F1** gộp đủ 39 mẫu rồi tính sáu lớp. Giữ cả hai số, không gọi chúng cùng một phép tổng hợp. Community ID được ghép với dataset/reference/tháng; không nối bằng tên nhóm đơn lẻ.


In [ ]:
from reddit_sna.reporting.tracking_demo import load_tracking_demo, build_tracking_figures, export_tracking_demo
PAIR = ROOT / "artifacts/detection-tracking/phuc-dang-final-20261020-001"
TRACKING_DEVELOPMENT = Path(os.environ.get("REDDIT_TRACKING_DEVELOPMENT", PAIR / "development")).resolve()
TRACKING_EVALUATION = Path(os.environ.get("REDDIT_TRACKING_EVALUATION", PAIR / "evaluation")).resolve()
tracking = load_tracking_demo(TRACKING_DEVELOPMENT, TRACKING_EVALUATION)
print("Tracking inputs:", TRACKING_DEVELOPMENT, TRACKING_EVALUATION)
display(HTML(table_html(tracking["summary"])))
print("Phúc tái lập output từ PR12; chưa xác nhận artifact/freeze lịch sử của Đăng.")


## 10. Confusion matrix sự kiện

Mỗi split có 39 nguồn truth, gồm K; support stable/grow/shrink/merge/split/dissolve là 21/3/3/6/3/3. Truth dùng ma trận 6×6, detector 6×7 với cột unmatched. Coverage mapping 1,0 không bảo đảm sự kiện đúng. Nhóm detector dư được thống kê riêng, không tạo sự kiện thứ bảy.


In [ ]:
tracking_figures = build_tracking_figures(tracking)
for figure in tracking_figures.values():
    plt.close(figure)
    display(figure)


## 11. Chỉ số từng lớp và trường hợp sai

Giữ mọi dự đoán sai từ cả hai split, không sửa nhãn để tăng điểm. Case dev-merge-easy-g11 nối phần detection khó ở trên với tracker: nhóm detector và liên kết có thể gây many-to-many; diagnostics ghi số nguồn mơ hồ. Chỉ dùng để mô tả sau khi xem kết quả, không chọn ngưỡng mới từ evaluation.


In [ ]:
display(HTML(table_html(tracking["per_class"])))
display(HTML(table_html(tracking["errors"], ["split", "dataset_id", "source_community_id", "truth_event", "predicted_event"])))
hard_tracking = TRACKING_DEVELOPMENT / "datasets/dev-merge-easy-g11/detector_tracking/tracker_diagnostics.json"
print("Tracker diagnostics case merge khó:")
display(JSON(json.loads(hard_tracking.read_text(encoding="utf-8"))))


## 12. Lưu phần tracking

Thư mục mới cùng cấp có đuôi `-tracking` chứa summary, 156 predictions, 24 dòng chỉ số từng lớp, trường hợp sai, bốn confusion CSV, hai hình PNG/SVG và metadata/hash. Hai số F1 khác cách tổng hợp được giữ rõ tên. Bằng chứng cũ và ZIP input T02 không bị ghi đè.


In [ ]:
tracking_metadata = export_tracking_demo(tracking, tracking_figures, TRACKING_OUTPUT)
print("Đã lưu tracking:", TRACKING_OUTPUT)
display(JSON({key: tracking_metadata[key] for key in ("status", "scope", "prediction_rows", "figure_count", "partner_acceptance")}))


## Giới hạn và bàn giao

Demo kỹ thuật trên dữ liệu tổng hợp gồm detection, tracking truth, tracking detector, mapping, AMI và F1. Chưa có kết quả forecasting hay Reddit từ notebook này. Notebook và kết quả tái lập do Phúc chuẩn bị, không thay xác nhận nghiệm thu của hai thành viên. Các run/freeze gốc của Đăng có thể bổ sung đối chiếu sau mà không cần đổi đầu vào T02 hoặc chạy lại detector.
